# 2. Feature Engineering - NYC TLC Data

This notebook performs feature engineering on the NYC TLC dataset to create meaningful features for ML models.

**Target Variable:** Trip duration (in minutes) - Regression problem

In [ ]:
# Import libraries
import yaml
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported!")

In [ ]:
# Initialize Spark
with open('../config/spark_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

spark_config = config['spark']

spark = SparkSession.builder \
    .appName("Feature_Engineering") \
    .master(spark_config['master']) \
    .config("spark.driver.memory", spark_config['driver_memory']) \
    .config("spark.executor.memory", spark_config['executor_memory']) \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

In [ ]:
# Load cleaned data
df = spark.read.parquet("../data/processed/nyc_tlc_clean")
print(f"Loaded {df.count():,} records")
df.printSchema()

## Create Target Variable: Trip Duration

In [ ]:
# Calculate trip duration in minutes
df = df.withColumn(
    "trip_duration_minutes",
    (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60
)

# Filter reasonable trip durations (1 min to 3 hours)
df = df.filter(
    (col("trip_duration_minutes") >= 1) &
    (col("trip_duration_minutes") <= 180)
)

print(f"Records after duration filtering: {df.count():,}")
df.select("trip_duration_minutes").describe().show()

## Temporal Features

In [ ]:
# Extract temporal features
df = df.withColumn("pickup_hour", hour("tpep_pickup_datetime")) \
       .withColumn("pickup_day", dayofweek("tpep_pickup_datetime")) \
       .withColumn("pickup_month", month("tpep_pickup_datetime")) \
       .withColumn("pickup_year", year("tpep_pickup_datetime"))

# Create time of day categories
df = df.withColumn(
    "time_of_day",
    when((col("pickup_hour") >= 6) & (col("pickup_hour") < 12), "morning")
    .when((col("pickup_hour") >= 12) & (col("pickup_hour") < 18), "afternoon")
    .when((col("pickup_hour") >= 18) & (col("pickup_hour") < 22), "evening")
    .otherwise("night")
)

# Weekend flag
df = df.withColumn(
    "is_weekend",
    when(col("pickup_day").isin([1, 7]), 1).otherwise(0)
)

# Rush hour flag (7-9 AM, 5-7 PM on weekdays)
df = df.withColumn(
    "is_rush_hour",
    when(
        ((col("pickup_hour").between(7, 9)) | (col("pickup_hour").between(17, 19))) &
        (col("is_weekend") == 0),
        1
    ).otherwise(0)
)

print("Temporal features created!")
df.select("pickup_hour", "pickup_day", "time_of_day", "is_weekend", "is_rush_hour").show(5)

## Distance and Speed Features

In [ ]:
# Calculate average speed (mph)
df = df.withColumn(
    "avg_speed_mph",
    (col("trip_distance") / (col("trip_duration_minutes") / 60))
)

# Distance categories
df = df.withColumn(
    "distance_category",
    when(col("trip_distance") < 2, "short")
    .when((col("trip_distance") >= 2) & (col("trip_distance") < 5), "medium")
    .when((col("trip_distance") >= 5) & (col("trip_distance") < 10), "long")
    .otherwise("very_long")
)

# Fare per mile
df = df.withColumn(
    "fare_per_mile",
    col("fare_amount") / col("trip_distance")
)

print("Distance and speed features created!")
df.select("trip_distance", "avg_speed_mph", "distance_category", "fare_per_mile").describe().show()

## Location Features

In [ ]:
# Same location flag (pickup and dropoff at same location)
df = df.withColumn(
    "same_location",
    when(col("PULocationID") == col("DOLocationID"), 1).otherwise(0)
)

# Popular pickup locations (top 10)
popular_pickup = df.groupBy("PULocationID").count() \
    .orderBy(desc("count")).limit(10) \
    .select("PULocationID").rdd.flatMap(lambda x: x).collect()

df = df.withColumn(
    "is_popular_pickup",
    when(col("PULocationID").isin(popular_pickup), 1).otherwise(0)
)

# Popular dropoff locations (top 10)
popular_dropoff = df.groupBy("DOLocationID").count() \
    .orderBy(desc("count")).limit(10) \
    .select("DOLocationID").rdd.flatMap(lambda x: x).collect()

df = df.withColumn(
    "is_popular_dropoff",
    when(col("DOLocationID").isin(popular_dropoff), 1).otherwise(0)
)

print("Location features created!")

## Payment and Fare Features

In [ ]:
# Total extras (extra + mta_tax + tolls + surcharges)
df = df.withColumn(
    "total_extras",
    col("extra") + col("mta_tax") + col("tolls_amount") + 
    col("improvement_surcharge") + coalesce(col("congestion_surcharge"), lit(0))
)

# Tip percentage
df = df.withColumn(
    "tip_percentage",
    (col("tip_amount") / col("fare_amount")) * 100
)

# High tipper flag (tip > 20%)
df = df.withColumn(
    "is_high_tipper",
    when(col("tip_percentage") > 20, 1).otherwise(0)
)

print("Payment features created!")
df.select("total_extras", "tip_percentage", "is_high_tipper").describe().show()

## Handle Missing Values

In [ ]:
# Fill missing passenger_count with median
median_passengers = df.approxQuantile("passenger_count", [0.5], 0.01)[0]
df = df.fillna({"passenger_count": median_passengers})

# Fill missing congestion_surcharge with 0
df = df.fillna({"congestion_surcharge": 0.0})

# Drop rows with any remaining nulls in key columns
key_columns = ["trip_distance", "fare_amount", "trip_duration_minutes", 
               "PULocationID", "DOLocationID"]
df = df.dropna(subset=key_columns)

print(f"Records after handling missing values: {df.count():,}")

## Feature Selection for ML

In [ ]:
# Select features for modeling
feature_columns = [
    # Numerical features
    "trip_distance", "passenger_count", "fare_amount",
    "pickup_hour", "pickup_day", "pickup_month",
    "avg_speed_mph", "fare_per_mile", "total_extras", "tip_percentage",
    
    # Binary features
    "is_weekend", "is_rush_hour", "same_location",
    "is_popular_pickup", "is_popular_dropoff", "is_high_tipper",
    
    # Categorical features (will be encoded)
    "VendorID", "RatecodeID", "payment_type",
    "PULocationID", "DOLocationID"
]

target_column = "trip_duration_minutes"

# Select only needed columns
df_features = df.select(feature_columns + [target_column])

print(f"Selected {len(feature_columns)} features")
print(f"Target: {target_column}")

## Cache and Persist

In [ ]:
# Cache the dataframe for faster access
df_features.cache()
print(f"Cached {df_features.count():,} records")

## Save Engineered Features

In [ ]:
# Save engineered features
output_path = "../data/processed/nyc_tlc_features"
df_features.write \
    .mode("overwrite") \
    .parquet(output_path)

print(f"Features saved to: {output_path}")

In [ ]:
# Save feature list for reference
import json

feature_metadata = {
    "numerical_features": [
        "trip_distance", "passenger_count", "fare_amount",
        "pickup_hour", "pickup_day", "pickup_month",
        "avg_speed_mph", "fare_per_mile", "total_extras", "tip_percentage"
    ],
    "binary_features": [
        "is_weekend", "is_rush_hour", "same_location",
        "is_popular_pickup", "is_popular_dropoff", "is_high_tipper"
    ],
    "categorical_features": [
        "VendorID", "RatecodeID", "payment_type",
        "PULocationID", "DOLocationID"
    ],
    "target": target_column
}

with open("../data/schemas/feature_metadata.json", "w") as f:
    json.dump(feature_metadata, f, indent=2)

print("Feature metadata saved!")

## Feature Statistics for Report

In [ ]:
# Generate feature statistics
print("\nFeature Statistics:")
df_features.describe().show()

# Correlation with target (sample for speed)
sample_pd = df_features.sample(fraction=0.01).toPandas()
correlations = sample_pd.corr()[target_column].sort_values(ascending=False)
print("\nTop correlations with target:")
print(correlations.head(10))

In [ ]:
print("Feature engineering complete!")